In [ ]:
# If running in Google Colab, uncomment:
!pip -q install gymnasium[classic-control] torch numpy matplotlib

# Imports + Global Config

In [ ]:
import math
import random
from dataclasses import dataclass
from collections import deque, namedtuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

import gymnasium as gym
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device: cpu


# Flags (key cell)

In [ ]:
# =========================
# Extensions switches
# =========================
USE_DOUBLE  = True
USE_DUELING = True
USE_NOISY   = False
USE_PER     = True

# =========================
# Environment
# =========================
ENV_ID = "CartPole-v1"
# You can later switch to:
# ENV_ID = "LunarLander-v3"

# =========================
# Training hyperparams
# =========================
GAMMA = 0.99
LR = 1e-3

BATCH_SIZE = 128
BUFFER_SIZE = 100_000
LEARNING_STARTS = 2_000

TARGET_UPDATE_EVERY = 1000      # hard update steps
SOFT_UPDATE = False
TAU = 0.005                     # if SOFT_UPDATE=True

TRAIN_FREQ = 1
GRAD_CLIP_NORM = 10.0

TOTAL_STEPS = 200_000
EVAL_EVERY = 10_000
EVAL_EPISODES = 5

# Epsilon-greedy only used when USE_NOISY=False
EPS_START = 1.0
EPS_END   = 0.05
EPS_DECAY_STEPS = 50_000

# PER params (used when USE_PER=True)
PER_ALPHA = 0.6
PER_BETA_START = 0.4
PER_BETA_END   = 1.0
PER_BETA_STEPS = 100_000
PER_EPS = 1e-6

# Env helper

In [ ]:
def make_env(env_id: str, seed: int):
    env = gym.make(env_id)
    env.reset(seed=seed)
    env.action_space.seed(seed)
    return env

env = make_env(ENV_ID, SEED)
obs_dim = env.observation_space.shape[0]
n_actions = env.action_space.n
print("obs_dim:", obs_dim, "n_actions:", n_actions)

obs_dim: 4 n_actions: 2


# Transition tuple

In [ ]:
Transition = namedtuple("Transition", ["s", "a", "r", "s2", "done"])

# Replay Buffers

## Uniform ReplayBuffer (baseline)

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity: int):
        self.buf = deque(maxlen=capacity)

    def __len__(self):
        return len(self.buf)

    def add(self, s, a, r, s2, done):
        self.buf.append(Transition(s, a, r, s2, done))

    def sample(self, batch_size: int):
        batch = random.sample(self.buf, batch_size)
        return self._to_tensors(batch), None, None  # (batch_tensors, idxs, weights)

    @staticmethod
    def _to_tensors(batch):
        s  = torch.tensor(np.array([b.s for b in batch]), dtype=torch.float32, device=device)
        a  = torch.tensor(np.array([b.a for b in batch]), dtype=torch.int64, device=device).unsqueeze(-1)
        r  = torch.tensor(np.array([b.r for b in batch]), dtype=torch.float32, device=device).unsqueeze(-1)
        s2 = torch.tensor(np.array([b.s2 for b in batch]), dtype=torch.float32, device=device)
        d  = torch.tensor(np.array([b.done for b in batch]), dtype=torch.float32, device=device).unsqueeze(-1)
        return s, a, r, s2, d

## PER: SumTree

In [ ]:
class SumTree:
    """
    Classic SumTree implementation (works for any capacity).
    Tree is stored as a flat array:
      - size = 2*capacity - 1
      - leaves are at indices [capacity-1 ... 2*capacity-2]
    """
    def __init__(self, capacity: int):
        self.capacity = capacity
        self.tree = np.zeros(2 * capacity - 1, dtype=np.float32)
        self.data = np.empty(capacity, dtype=object)
        self.write = 0
        self.size = 0

    def total(self) -> float:
        return float(self.tree[0])

    def add(self, p: float, data):
        leaf_idx = self.write + (self.capacity - 1)
        self.data[self.write] = data
        self.update(leaf_idx, p)

        self.write = (self.write + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def update(self, idx: int, p: float):
        change = p - self.tree[idx]
        self.tree[idx] = p

        # propagate change up to root
        while idx != 0:
            idx = (idx - 1) // 2
            self.tree[idx] += change

    def get(self, s: float):
        """
        Find the leaf such that the cumulative sum covers s
        """
        idx = 0  # start from root
        while True:
            left = 2 * idx + 1
            right = left + 1
            if left >= len(self.tree):  # idx is a leaf
                break

            if s <= self.tree[left]:
                idx = left
            else:
                s -= self.tree[left]
                idx = right

        data_idx = idx - (self.capacity - 1)
        return idx, float(self.tree[idx]), self.data[data_idx]

# PER Buffer

In [ ]:
class PrioritizedReplayBuffer:
    def __init__(self, capacity: int, alpha: float):
        self.alpha = alpha
        self.tree = SumTree(capacity)
        self.max_priority = 1.0

    def __len__(self):
        return self.tree.size

    def add(self, s, a, r, s2, done):
        data = Transition(s, a, r, s2, done)
        p = (self.max_priority + PER_EPS) ** self.alpha
        self.tree.add(p, data)

    def sample(self, batch_size: int, beta: float):
        total = self.tree.total()
        if total <= 0:
            raise RuntimeError("SumTree total priority is zero; cannot sample.")

        batch = []
        idxs = []
        priorities = []

        segment = total / batch_size

        for i in range(batch_size):
            a = segment * i
            b = segment * (i + 1)
            s = random.uniform(a, b)

            idx, p, data = self.tree.get(s)

            # Safety check: should not happen after SumTree fix, but keep it robust
            if data is None:
                # fallback: resample uniformly in [0,total]
                s = random.uniform(0.0, total)
                idx, p, data = self.tree.get(s)

            batch.append(data)
            idxs.append(idx)
            priorities.append(p)

        probs = np.array(priorities, dtype=np.float32) / (total + 1e-8)
        probs = np.clip(probs, 1e-8, None)  # avoid zeros

        weights = (len(self) * probs) ** (-beta)
        weights = weights / (weights.max() + 1e-8)
        weights_t = torch.tensor(weights, dtype=torch.float32, device=device).unsqueeze(-1)

        return ReplayBuffer._to_tensors(batch), idxs, weights_t

    def update_priorities(self, idxs, priorities):
        for idx, p in zip(idxs, priorities):
            p = max(float(p), PER_EPS)  # ensure > 0
            self.max_priority = max(self.max_priority, p)
            self.tree.update(idx, (p ** self.alpha))

# Networks: Vanilla / Dueling / Noisy

## Noisy Linear (Factorized Gaussian)

In [ ]:
class NoisyLinear(nn.Module):
    def __init__(self, in_features, out_features, sigma_init=0.5):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        self.weight_mu = nn.Parameter(torch.empty(out_features, in_features))
        self.weight_sigma = nn.Parameter(torch.empty(out_features, in_features))
        self.register_buffer("weight_eps", torch.empty(out_features, in_features))

        self.bias_mu = nn.Parameter(torch.empty(out_features))
        self.bias_sigma = nn.Parameter(torch.empty(out_features))
        self.register_buffer("bias_eps", torch.empty(out_features))

        self.sigma_init = sigma_init
        self.reset_parameters()
        self.reset_noise()

    def reset_parameters(self):
        mu_range = 1 / math.sqrt(self.in_features)
        self.weight_mu.data.uniform_(-mu_range, mu_range)
        self.bias_mu.data.uniform_(-mu_range, mu_range)

        self.weight_sigma.data.fill_(self.sigma_init / math.sqrt(self.in_features))
        self.bias_sigma.data.fill_(self.sigma_init / math.sqrt(self.out_features))

    @staticmethod
    def _f(x):
        return torch.sign(x) * torch.sqrt(torch.abs(x))

    def reset_noise(self):
        eps_in = torch.randn(self.in_features, device=self.weight_mu.device)
        eps_out = torch.randn(self.out_features, device=self.weight_mu.device)
        eps_in = self._f(eps_in)
        eps_out = self._f(eps_out)

        self.weight_eps.copy_(eps_out.unsqueeze(1) * eps_in.unsqueeze(0))
        self.bias_eps.copy_(eps_out)

    def forward(self, x):
        if self.training:
            w = self.weight_mu + self.weight_sigma * self.weight_eps
            b = self.bias_mu + self.bias_sigma * self.bias_eps
        else:
            w = self.weight_mu
            b = self.bias_mu
        return F.linear(x, w, b)

## QNetwork builder (Dueling + Noisy optional)

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, obs_dim, n_actions, dueling=False, noisy=False, hidden=256):
        super().__init__()
        self.dueling = dueling
        self.noisy = noisy

        Linear = NoisyLinear if noisy else nn.Linear

        self.feature = nn.Sequential(
            Linear(obs_dim, hidden),
            nn.ReLU(),
            Linear(hidden, hidden),
            nn.ReLU(),
        )

        if dueling:
            self.value = nn.Sequential(
                Linear(hidden, hidden),
                nn.ReLU(),
                Linear(hidden, 1)
            )
            self.adv = nn.Sequential(
                Linear(hidden, hidden),
                nn.ReLU(),
                Linear(hidden, n_actions)
            )
        else:
            self.head = nn.Sequential(
                Linear(hidden, hidden),
                nn.ReLU(),
                Linear(hidden, n_actions)
            )

    def reset_noise(self):
        if not self.noisy:
            return
        for m in self.modules():
            if isinstance(m, NoisyLinear):
                m.reset_noise()

    def forward(self, x):
        z = self.feature(x)
        if self.dueling:
            v = self.value(z)                  # (B,1)
            a = self.adv(z)                    # (B,A)
            a = a - a.mean(dim=1, keepdim=True)
            return v + a
        else:
            return self.head(z)

# Agent: action selection + training step

## Epsilon schedule (only if not Noisy)

In [ ]:
def epsilon_by_step(step):
    if step >= EPS_DECAY_STEPS:
        return EPS_END
    frac = step / EPS_DECAY_STEPS
    return EPS_START + frac * (EPS_END - EPS_START)

## Create networks, optimizer, buffer

In [ ]:
q_online = QNetwork(obs_dim, n_actions, dueling=USE_DUELING, noisy=USE_NOISY).to(device)
q_target = QNetwork(obs_dim, n_actions, dueling=USE_DUELING, noisy=USE_NOISY).to(device)
q_target.load_state_dict(q_online.state_dict())
q_target.eval()

optimizer = optim.Adam(q_online.parameters(), lr=LR)

if USE_PER:
    buffer = PrioritizedReplayBuffer(BUFFER_SIZE, alpha=PER_ALPHA)
else:
    buffer = ReplayBuffer(BUFFER_SIZE)

print(q_online)

QNetwork(
  (feature): Sequential(
    (0): Linear(in_features=4, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
  )
  (value): Sequential(
    (0): Linear(in_features=256, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=1, bias=True)
  )
  (adv): Sequential(
    (0): Linear(in_features=256, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=2, bias=True)
  )
)


## One training step (Double / PER / Noisy-aware)

In [ ]:
def compute_beta(step):
    if step >= PER_BETA_STEPS:
        return PER_BETA_END
    frac = step / PER_BETA_STEPS
    return PER_BETA_START + frac * (PER_BETA_END - PER_BETA_START)

@torch.no_grad()
def td_target_double(s2, r, done):
    # Double DQN target
    # a* = argmax Q_online(s2)
    a2 = q_online(s2).argmax(dim=1, keepdim=True)
    q2 = q_target(s2).gather(1, a2)
    return r + GAMMA * (1.0 - done) * q2

@torch.no_grad()
def td_target_vanilla(s2, r, done):
    # Vanilla DQN target: max_a Q_target(s2,a)
    q2 = q_target(s2).max(dim=1, keepdim=True).values
    return r + GAMMA * (1.0 - done) * q2

def train_step(step):
    if len(buffer) < max(LEARNING_STARTS, BATCH_SIZE):
        return None

    if USE_PER:
        beta = compute_beta(step)
        (s, a, r, s2, done), idxs, weights = buffer.sample(BATCH_SIZE, beta=beta)
    else:
        (s, a, r, s2, done), idxs, weights = buffer.sample(BATCH_SIZE)
        weights = torch.ones((BATCH_SIZE, 1), device=device)

    # Noisy nets: refresh noise per step
    if USE_NOISY:
        q_online.reset_noise()
        q_target.reset_noise()

    q = q_online(s).gather(1, a)

    with torch.no_grad():
        if USE_DOUBLE:
            y = td_target_double(s2, r, done)
        else:
            y = td_target_vanilla(s2, r, done)

    td_err = (y - q)  # (B,1)

    loss = (weights * td_err.pow(2)).mean()

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(q_online.parameters(), GRAD_CLIP_NORM)
    optimizer.step()

    # PER priority update
    if USE_PER:
        prios = td_err.detach().abs().squeeze(-1).cpu().numpy() + PER_EPS
        buffer.update_priorities(idxs, prios)

    # Target network update
    if SOFT_UPDATE:
        with torch.no_grad():
            for p_t, p in zip(q_target.parameters(), q_online.parameters()):
                p_t.data.mul_(1 - TAU).add_(TAU * p.data)
    else:
        if step % TARGET_UPDATE_EVERY == 0:
            q_target.load_state_dict(q_online.state_dict())

    return float(loss.item())

## Action selection (ε-greedy or Noisy)

In [ ]:
@torch.no_grad()
def select_action(obs, step):
    obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)

    if USE_NOISY:
        # stochasticity comes from noisy layers
        q_online.reset_noise()
        qvals = q_online(obs_t)
        return int(qvals.argmax(dim=1).item())

    eps = epsilon_by_step(step)
    if random.random() < eps:
        return env.action_space.sample()
    qvals = q_online(obs_t)
    return int(qvals.argmax(dim=1).item())

# Training loop + evaluation

## Evaluation helper

In [ ]:
@torch.no_grad()
def evaluate(env_id, episodes=5):
    e = make_env(env_id, SEED + 999)
    returns = []
    for _ in range(episodes):
        obs, _ = e.reset()
        done = False
        total = 0.0
        while not done:
            obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            qvals = q_online(obs_t)
            a = int(qvals.argmax(dim=1).item())
            obs, r, terminated, truncated, _ = e.step(a)
            done = terminated or truncated
            total += r
        returns.append(total)
    e.close()
    return float(np.mean(returns)), float(np.std(returns))

## Train

In [ ]:
obs, _ = env.reset()
episode_return = 0.0
episode_len = 0
episode = 0

losses = []
eval_steps = []
eval_means = []
eval_stds = []

for step in range(1, TOTAL_STEPS + 1):
    a = select_action(obs, step)
    obs2, r, terminated, truncated, _ = env.step(a)
    done = terminated or truncated

    buffer.add(obs, a, r, obs2, done)

    obs = obs2
    episode_return += r
    episode_len += 1

    if step % TRAIN_FREQ == 0:
        loss = train_step(step)
        if loss is not None:
            losses.append(loss)

    if done:
        obs, _ = env.reset()
        episode += 1
        episode_return = 0.0
        episode_len = 0

    if step % EVAL_EVERY == 0:
        mean_r, std_r = evaluate(ENV_ID, episodes=EVAL_EPISODES)
        eval_steps.append(step)
        eval_means.append(mean_r)
        eval_stds.append(std_r)
        print(f"[step {step:7d}] eval return: {mean_r:.1f} ± {std_r:.1f} | buffer: {len(buffer)} | last_loss: {losses[-1] if losses else None}")

[step   10000] eval return: 354.0 ± 61.3 | buffer: 10000 | last_loss: 0.017633259296417236
[step   20000] eval return: 124.4 ± 2.8 | buffer: 20000 | last_loss: 0.0402907058596611
[step   30000] eval return: 166.8 ± 20.8 | buffer: 30000 | last_loss: 0.08672279119491577
[step   40000] eval return: 223.4 ± 18.1 | buffer: 40000 | last_loss: 0.03897491842508316
[step   50000] eval return: 500.0 ± 0.0 | buffer: 50000 | last_loss: 0.13630223274230957
[step   60000] eval return: 158.8 ± 9.2 | buffer: 60000 | last_loss: 0.11693886667490005
[step   70000] eval return: 456.4 ± 87.2 | buffer: 70000 | last_loss: 0.0879674181342125
[step   80000] eval return: 182.2 ± 18.2 | buffer: 80000 | last_loss: 0.2204715460538864
[step   90000] eval return: 150.0 ± 4.0 | buffer: 90000 | last_loss: 0.4386216402053833
[step  100000] eval return: 500.0 ± 0.0 | buffer: 100000 | last_loss: 0.05336274206638336
[step  110000] eval return: 500.0 ± 0.0 | buffer: 100000 | last_loss: 0.18012525141239166
[step  120000] ev

## Plots

In [ ]:
plt.figure()
plt.plot(losses)
plt.title("Training loss")
plt.xlabel("update step")
plt.ylabel("loss")
plt.show()

plt.figure()
plt.plot(eval_steps, eval_means)
plt.fill_between(eval_steps, np.array(eval_means)-np.array(eval_stds), np.array(eval_means)+np.array(eval_stds), alpha=0.2)
plt.title("Evaluation return")
plt.xlabel("environment step")
plt.ylabel("return")
plt.show()